In [63]:
from dotenv import load_dotenv
load_dotenv()

True

In [64]:
import os

import pandas as pd
import mlflow
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    OneHotEncoder, 
    SplineTransformer, 
    QuantileTransformer, 
    RobustScaler,
    PolynomialFeatures,
    KBinsDiscretizer,
)

TABLE_NAME = "clean_users_churn"

TRACKING_SERVER_HOST = "127.0.0.1"
TRACKING_SERVER_PORT = 5000

EXPERIMENT_NAME = "churn_simonreise"
RUN_NAME = "preprocessing" 
REGISTRY_MODEL_NAME = "churn_model_simonreise_catboost_basic"

In [79]:
import psycopg

connection = {"sslmode": "require", "target_session_attrs": "read-write"}
postgres_credentials = {
    "host": os.getenv("DB_DESTINATION_HOST"),
    "port": os.getenv("DB_DESTINATION_PORT"),
    "dbname": os.getenv("DB_DESTINATION_NAME"),
    "user": os.getenv("DB_DESTINATION_USER"),
    "password": os.getenv("DB_DESTINATION_PASSWORD"),
}

connection.update(postgres_credentials)

with psycopg.connect(**connection) as conn:

    with conn.cursor() as cur:
        cur.execute(f"SELECT * FROM {TABLE_NAME}")
        data = cur.fetchall()
        columns = [col[0] for col in cur.description]

df = pd.DataFrame(data, columns=columns)

df.head(2) 

,id,customer_id,begin_date,end_date,type,paperless_billing,payment_method,monthly_charges,total_charges,internet_service,...,device_protection,tech_support,streaming_tv,streaming_movies,gender,senior_citizen,partner,dependents,multiple_lines,target
0,1,9763-GRSKD,2019-01-01,NaT,Month-to-month,Yes,Mailed check,49.95,587.45,DSL,...,No,No,No,No,Male,0,Yes,Yes,No,0
1,2,8402-OOOHJ,2016-09-01,NaT,Two year,No,Mailed check,20.65,835.15,Fiber optic,...,No,No,No,No,Female,0,No,No,No,0


In [80]:
obj_df = df.select_dtypes(include="object")

In [82]:
# определение категориальных колонок, которые будут преобразованы
cat_columns = ["type", "payment_method", "internet_service", "gender"]

# создание объекта OneHotEncoder для преобразования категориальных переменных
# auto - автоматическое определение категорий
# ignore - игнорировать ошибки, если встречается неизвестная категория
# max_categories - максимальное количество уникальных категорий
# sparse_output - вывод в виде разреженной матрицы, если False, то в виде обычного массива
# drop="first" - удаляет первую категорию, чтобы избежать ловушки мультиколлинеарности
encoder_oh = OneHotEncoder(categories="auto", max_categories=10, handle_unknown="ignore", drop="first", sparse_output=False)

# применение OneHotEncoder к данным. Преобразование категориальных данных в массив
encoded_features = encoder_oh.fit_transform(obj_df)
# преобразование полученных признаков в DataFrame и установка названий колонок
# get_feature_names_out() - получение имён признаков после преобразования
encoded_df = pd.DataFrame(encoded_features, columns=encoder_oh.get_feature_names_out())

# конкатенация исходного DataFrame с новым DataFrame, содержащим закодированные категориальные признаки
# axis=1 означает конкатенацию по колонкам
obj_df = pd.concat([obj_df, encoded_df], axis=1)

obj_df.head(2)

,customer_id,type,paperless_billing,payment_method,internet_service,online_security,online_backup,device_protection,tech_support,streaming_tv,...,online_security_Yes_1.0,online_backup_Yes_1.0,device_protection_Yes_1.0,tech_support_Yes_1.0,streaming_tv_Yes_1.0,streaming_movies_Yes_1.0,gender_Male_1.0,partner_Yes_1.0,dependents_Yes_1.0,multiple_lines_Yes_1.0
0,9763-GRSKD,Month-to-month,Yes,Mailed check,DSL,Yes,No,No,No,No,...,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0
1,8402-OOOHJ,Two year,No,Mailed check,Fiber optic,No,No,No,No,No,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [83]:
num_columns = ["monthly_charges", "total_charges"]
num_df = df[num_columns]

n_knots = 3
degree_spline = 4
n_quantiles=100
degree = 3
n_bins = 5
encode = 'ordinal'
strategy = 'uniform'
subsample = None


# SplineTransformer
encoder_spl = SplineTransformer(n_knots=n_knots, degree=degree_spline)
encoded_features = encoder_spl.fit_transform(df[num_columns])

encoded_df = pd.DataFrame(encoded_features, columns=encoder_spl.get_feature_names_out())
num_df = pd.concat([num_df, encoded_df], axis=1)


# QuantileTransformer
encoder_q = QuantileTransformer(n_quantiles=n_quantiles)
encoded_features = encoder_q.fit_transform(df[num_columns])

encoded_df = pd.DataFrame(encoded_features, columns=encoder_q.get_feature_names_out())
encoded_df.columns = [col + f"_q_{n_quantiles}" for col in num_columns]
num_df = pd.concat([num_df, encoded_df], axis=1)


# RobustScaler
encoder_rb = RobustScaler()
encoded_features = encoder_rb.fit_transform(df[num_columns])

encoded_df = pd.DataFrame(encoded_features, columns=encoder_rb.get_feature_names_out())
encoded_df.columns = [col + f"_robust" for col in num_columns]
num_df = pd.concat([num_df, encoded_df], axis=1)


# PolynomialFeatures
encoder_pol = PolynomialFeatures(degree=degree)
encoded_features = encoder_pol.fit_transform(df[num_columns])

encoded_df = pd.DataFrame(encoded_features, columns=encoder_pol.get_feature_names_out())
# get all columns after the intercept and original features
encoded_df = encoded_df.iloc[:, 1 + len(num_columns):]
encoded_df.columns = encoder_pol.get_feature_names_out()[1 + len(num_columns):]
num_df = pd.concat([num_df, encoded_df], axis=1)

# KBinsDiscretizer
encoder_kbd = KBinsDiscretizer(n_bins=n_bins, encode=encode, strategy=strategy, subsample=subsample)
encoded_features = encoder_kbd.fit_transform(df[num_columns])

encoded_df = pd.DataFrame(encoded_features, columns=encoder_kbd.get_feature_names_out())
encoded_df.columns = [col + f"_bin" for col in num_columns]
num_df = pd.concat([num_df, encoded_df], axis=1)


num_df.head(2)


,monthly_charges,total_charges,monthly_charges_sp_0,monthly_charges_sp_1,monthly_charges_sp_2,monthly_charges_sp_3,monthly_charges_sp_4,monthly_charges_sp_5,total_charges_sp_0,total_charges_sp_1,...,total_charges_robust,monthly_charges^2,monthly_charges total_charges,total_charges^2,monthly_charges^3,monthly_charges^2 total_charges,monthly_charges total_charges^2,total_charges^3,monthly_charges_bin,total_charges_bin
0,49.95,587.45,0.000774,0.142550,0.588331,0.261746,6.599052e-03,0.0,0.023735,0.389490,...,-0.242592,2495.0025,29343.1275,345097.5025,124625.374875,1.465689e+06,1.723762e+07,2.027275e+08,1.0,0.0
1,20.65,835.15,0.034259,0.433936,0.481590,0.050214,2.168151e-07,0.0,0.018078,0.358392,...,-0.169561,426.4225,17245.8475,697475.5225,8805.624625,3.561268e+05,1.440287e+07,5.824967e+08,0.0,0.0


In [84]:
numeric_transformer = ColumnTransformer(transformers=[
		("spl", encoder_spl, num_columns),
        ("q", encoder_q, num_columns),
        ("rb", encoder_rb, num_columns),
        ("pol", encoder_pol, num_columns),
        ("kbd", encoder_kbd, num_columns),
	]
)

categorical_transformer = Pipeline(steps=[
	    ("encoder", encoder_oh),
    ]
)

preprocessor = ColumnTransformer(transformers=[
	    ("num", numeric_transformer, num_columns),
        ("cat", categorical_transformer, cat_columns),
    ], n_jobs=-1,
)

encoded_features = preprocessor.fit_transform(df)

transformed_df = pd.DataFrame(encoded_features, columns=preprocessor.get_feature_names_out())

df = pd.concat([df, transformed_df], axis=1)
df.head(2)

,id,customer_id,begin_date,end_date,type,paperless_billing,payment_method,monthly_charges,total_charges,internet_service,...,num__pol__total_charges^3,num__kbd__monthly_charges,num__kbd__total_charges,cat__type_One year,cat__type_Two year,cat__payment_method_Credit card (automatic),cat__payment_method_Electronic check,cat__payment_method_Mailed check,cat__internet_service_Fiber optic,cat__gender_Male
0,1,9763-GRSKD,2019-01-01,NaT,Month-to-month,Yes,Mailed check,49.95,587.45,DSL,...,2.027275e+08,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0
1,2,8402-OOOHJ,2016-09-01,NaT,Two year,No,Mailed check,20.65,835.15,Fiber optic,...,5.824967e+08,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0


In [85]:
preprocessor

,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,-1
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,n_knots,3
,degree,4
,knots,'uniform'


In [54]:
os.environ["MLFLOW_S3_ENDPOINT_URL"] = "https://storage.yandexcloud.net"
os.environ["AWS_ACCESS_KEY_ID"] = os.getenv("AWS_ACCESS_KEY_ID")
os.environ["AWS_SECRET_ACCESS_KEY"] = os.getenv("AWS_SECRET_ACCESS_KEY")

mlflow.set_tracking_uri(f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}")
mlflow.set_registry_uri(f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}")

experiment_id = mlflow.get_experiment_by_name(EXPERIMENT_NAME).experiment_id

with mlflow.start_run(run_name=RUN_NAME, experiment_id=experiment_id) as run:
    run_id = run.info.run_id

    mlflow.sklearn.log_model(preprocessor, "column_transformer") 

2026/03/07 08:21:31 WARNING mlflow.sklearn: Model was missing function: predict. Not logging python_function flavor!


In [55]:
run_id

'4152ee89475a47a2b86ec2275fb0c5e7'

In [86]:
from catboost import CatBoostClassifier

In [87]:
df = df.drop(columns=["id", "end_date", "customer_id"])

In [88]:
model = CatBoostClassifier(auto_class_weights='Balanced')

In [97]:
model.fit(df.drop(columns=["target"]), df["target"], cat_features=df.select_dtypes(include=["object"]).columns.tolist())

Learning rate set to 0.023675
0:	learn: 0.6825597	total: 73.4ms	remaining: 1m 13s
1:	learn: 0.6703049	total: 87.7ms	remaining: 43.8s
2:	learn: 0.6608331	total: 102ms	remaining: 33.8s
3:	learn: 0.6512929	total: 115ms	remaining: 28.7s
4:	learn: 0.6430910	total: 129ms	remaining: 25.8s
5:	learn: 0.6325470	total: 144ms	remaining: 23.8s
6:	learn: 0.6245581	total: 159ms	remaining: 22.6s
7:	learn: 0.6151093	total: 173ms	remaining: 21.4s
8:	learn: 0.6084784	total: 188ms	remaining: 20.7s
9:	learn: 0.6024058	total: 202ms	remaining: 20s
10:	learn: 0.5946163	total: 216ms	remaining: 19.5s
11:	learn: 0.5886462	total: 230ms	remaining: 19s
12:	learn: 0.5809660	total: 245ms	remaining: 18.6s
13:	learn: 0.5762870	total: 260ms	remaining: 18.3s
14:	learn: 0.5711174	total: 277ms	remaining: 18.2s
15:	learn: 0.5647712	total: 291ms	remaining: 17.9s
16:	learn: 0.5601078	total: 304ms	remaining: 17.6s
17:	learn: 0.5565149	total: 318ms	remaining: 17.4s
18:	learn: 0.5525235	total: 332ms	remaining: 17.1s
19:	learn: 0

In [98]:
X_test = df.drop(columns=["target"])
y_test = df["target"]
prediction = model.predict(X_test)
probas = model.predict_proba(X_test)

In [99]:
from sklearn.metrics import confusion_matrix, roc_auc_score, precision_score, recall_score, f1_score, log_loss
# импортируйте необходимые вам модули

# заведите словарь со всеми метриками
metrics = {}

# посчитайте метрики из модуля sklearn.metrics
# err_1 — ошибка первого рода
# err_2 — ошибка второго рода
_, err1, _, err2 = confusion_matrix(y_test, prediction, normalize="all").ravel()
auc = roc_auc_score(y_test, probas[:, 1])
precision = precision_score(y_test, prediction)
recall = recall_score(y_test, prediction)
f1 = f1_score(y_test, prediction)
logloss = log_loss(y_test, prediction)

# запишите значения метрик в словарь
metrics["err1"] = err1
metrics["err2"] = err2
metrics["auc"] = auc
metrics["precision"] = precision
metrics["recall"] = recall
metrics["f1"] = f1
metrics["logloss"] = logloss

In [100]:
pip_requirements = "./requirements.txt"
signature = mlflow.models.infer_signature(X_test, prediction)
input_example = X_test[:10]
metadata = {"model_type": "monthly"}

/home/mle-user/mle_projects/mle-mlflow/.venv/lib/python3.10/site-packages/mlflow/models/signature.py:212: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  inputs = _infer_schema(model_input) if model_input is not None else None


In [101]:
RUN_NAME = "preprocessing"

experiment_id = mlflow.get_experiment_by_name(EXPERIMENT_NAME).experiment_id

with mlflow.start_run(run_name=RUN_NAME, experiment_id=experiment_id) as run:
    run_id = run.info.run_id
    # ваш код здесь
    model_info = mlflow.catboost.log_model(
        cb_model=model,
        artifact_path="models",
        pip_requirements=pip_requirements,
        signature=signature,
        input_example=input_example,
        metadata=metadata,
        registered_model_name=REGISTRY_MODEL_NAME,
        await_registration_for=60,
    )
    mlflow.log_metrics(metrics)

Registered model 'churn_model_simonreise_catboost_basic' already exists. Creating a new version of this model...
2026/03/07 08:43:33 INFO mlflow.tracking._model_registry.client: Waiting up to 60 seconds for model version to finish creation. Model name: churn_model_simonreise_catboost_basic, version 4
Created version '4' of model 'churn_model_simonreise_catboost_basic'.
